In [2]:
using ITensors
using ITensorMPS
using Random
using LinearAlgebra
using Statistics
using PrettyTables

#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end

random_cdw_state (generic function with 1 method)

2-rdm

In [ ]:
function spin_correlators_mpo(state, sites, i, j, l, k)
    spins = ["up", "dn"]

    matrix_block = zeros(ComplexF64, 4, 4)

    for i_spin in 1:2
        for j_spin in 1:2
            
            row = 2 * (i_spin - 1) + j_spin

            for l_spin in 1:2
                for k_spin in 1:2

                    col = 2 * (l_spin - 1) + k_spin

                    ampo_op = OpSum()

                    ampo_op += ("Cdag" * spins[i_spin], i, "Cdag" * spins[j_spin], j, "C" * spins[l_spin], l, "C" * spins[k_spin], k)

                    O_mpo = MPO(ampo_op, sites)
                        
                    matrix_block[row, col] = inner(prime(state), O_mpo, state)

                    # ampo_op = nothing
                    # O_mpo = nothing
                    # GC.gc()
                end
            end
        end
    end
    @show i, j, k, l
    # @pt  :header = ["↑↑", "↑↓", "↓↑", "↓↓"] matrix_block :header = ["↑↑", "↑↓", "↓↑", "↓↓"]
    return matrix_block
end
function build_2_particle_rdm(state, sites)

    L = length(state)
    rho_2 = zeros(ComplexF64, (2*L)^2, (2*L)^2)
    
    # (L^2) x (L^2) blocks of spin correlators.
    for i in 1:L 
        for j in 1:L 
            for l in 1:L 
                for k in 1:L
                    
                    block_spin_correlators = spin_correlators_mpo(state, sites, i, j, l, k)

                    row_block = 4 *((i-1)* L + (j-1))
                    col_block = 4 *((l-1)* L + (k-1))

                    rho_2[row_block+1:row_block+4, col_block+1:col_block+4] = block_spin_correlators
                    
                    @show row_block + 4, col_block + 4
                    # @pt rho_2
                end
            end
        end
    end
    rho_2 = - rho_2 / (L*(L-1))
end

In [17]:
L = 3
sites = siteinds("Electron", L; conserve_qns=true)

Npart = floor(Int, L/2)
Nup = Npart + L % 2 
Ndn = L - Nup 

1

In [ ]:
state = random_metallic_state(L, Nup, Ndn)

psi = random_mps(sites, state, linkdims=4) 

In [ ]:
rho_2 = build_2_particle_rdm(psi, sites)
@show tr(rho_2)
@show ishermitian(rho_2)

In [ ]:
i = 1
j = 1
l = 1
k = 1

L = 4

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

sites = siteinds("Electron", L; conserve_qns=true)

state = random_metallic_state(L, Nup, Ndn)

psi = random_mps(sites, state; linkdims=10)

block = spin_correlators_mpo(psi, sites, i, j, l, k)

println("tr ρ = ", tr(block))
println("is hermitian ? ", ishermitian(block))
#
#lambdas = eigvals(block)
#
#@show real(lambdas)
#@show sum(real(lambdas))

In [ ]:
function two_fermion_basis_pairs(L)
    num_modes = 2 * L # Total number of single-particle modes
    pairs = []
    for m1 in 1:num_modes
        for m2 in (m1 + 1):num_modes # Ensures m1 < m2 and m1 != m2
            push!(pairs, (m1, m2))
        end
    end
    return pairs
end

function spin(site)
    return isodd((site + 1) % 2) ? "up" : "dn"
end
function mode_to_site_spin(m::Int)
    site = (m + 1) ÷ 2
    spin = isodd(m) ? "up" : "dn"
    return site, spin
end

function build_2_particle_rdm(state, sites)
    L = length(state)

    pairs = two_fermion_basis_indices(L) 
    
    dim = length(pairs)

    @show dim
    
    rho_2 = zeros(ComplexF64, dim, dim)
    
    spins = ["up", "dn"]
    # (L^2) x (L^2) blocks of spin correlators.
    for (p, (i, j)) in enumerate(pairs)
        for (q, (k, l)) in enumerate(pairs)
                    
            # block_spin_correlators = spin_correlators_mpo(state, sites, i, j, l, k)
            ampo_op = AutoMPO()
            
            # i_spin = isodd(i) ? "up" : "dn" 
            # j_spin = isodd(j) ? "up" : "dn" 
            # k_spin = isodd(k) ? "up" : "dn" 
            # l_spin = isodd(l) ? "up" : "dn" 
            
            @show (p, q), (i, j), (k, l)
            @show (i, spin(i)), (j, spin(j)), (k, spin(k)), (l, spin(l))
            
            ampo_op += ("Cdag" * spin(i), i, "Cdag" * spin(j), j, "C" * spin(l), l, "C" * spin(k), k)
            
            O_mpo = MPO(ampo_op, sites)
                            
            rho_2[p, q] = inner(prime(state), O_mpo, state)
        end
    end
    @show tr(rho_2)
    return rho_2 
end

two_fermion_basis_pairs (generic function with 2 methods)

In [1]:
L = 5

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

sites = siteinds("Electron", L; conserve_qns=true)

state_ = random_metallic_state(L, Nup, Ndn)

psi = random_mps(sites, state_; linkdims=10)

rho_2_matrix = build_2_particle_rdm(psi, sites)

UndefVarError: UndefVarError: `siteinds` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [3]:
function build_2_particle_rdm(phi, sites)

        L = length(phi)

        pairs = two_fermion_basis_pairs(L)

        dim = length(pairs)

        @show dim

        rho_2 = zeros(ComplexF64, dim, dim)

        for (p, (i, j)) in enumerate(pairs)
            @show (p, (i, j))

            i_site, i_spin = mode_to_site_spin(i)
            j_site, i_spin = mode_to_site_spin(i)
            j_site, j_spin = mode_to_site_spin(j)

            @show (mode_to_site_spin(i), mode_to_site_spin(j))

           for (q, (k, l)) in enumerate(pairs)

               auto_op = AutoMPO()

               @show (q, (k, l))

               k_site, k_spin = mode_to_site_spin(k)
               l_site, l_spin = mode_to_site_spin(l)

               @show (mode_to_spin(k), mode_to_site_spin(l))

                auto_op += ("Cdag"*i_spin,i_site, "Cdag"*j_spin,j_spin, "C"*k_spin,k_site, "C"*l_spin,l_site)

               O_mpo = MPO(auto_op, sites)

               rho_2[p, q] = inner(prime(phi), O_mpo, phi)
            end
        end
        @pt rho_2
end

build_2_particle_rdm (generic function with 1 method)